# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aamnamalik16-bit/flyrank-ML-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os

if not os.path.exists('flyrank-ML-internship'):
    !git clone https://github.com/aamnamalik16-bit/flyrank-ML-internship.git

os.chdir('flyrank-ML-internship')

!pip install duckdb -q

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
print("Setup complete!")

Cloning into 'flyrank-ML-internship'...
remote: Enumerating objects: 178, done.
remote: Counting objects: 100% (178/178), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 178 (delta 84), reused 111 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (178/178), 1.87 MiB | 4.67 MiB/s, done.
Resolving deltas: 100% (84/84), done.
Setup complete!


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: Random Forest achieves Precision@50 = 0.740, a ~3x lift over the rule baseline (0.240)

Methodology question: The label is_declining_label is derived from trend_direction, which is computed from trend_pct — all from the same 90-day snapshot. This means the label and features share the same time window. A stronger validation design would define features from an earlier window (e.g. days 0–60) and the label from a later window (e.g. days 61–90), so the model is genuinely predicting future movement rather than describing the present state.

Finding 2: Client-holdout validation was used to prevent data leakage across clients

Methodology question: Client holdout is the right idea — it prevents the model from memorizing client-specific patterns. However, the paper does not report how many clients were held out or whether the holdout clients are representative of the full distribution. If the 20% holdout happened to contain easier or harder clients, the reported Precision@50 could be optimistic or pessimistic. Reporting results across multiple holdout splits (cross-validation by client) would make the claim more robust.

In [2]:
print("Finding 1: label and features share the same 90-day window")
print("Finding 2: client holdout size and representativeness not reported")
print("Both questions are constructive — not criticism, but next-level rigor")

Finding 1: label and features share the same 90-day window
Finding 2: client holdout size and representativeness not reported
Both questions are constructive — not criticism, but next-level rigor


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Before/after comparison: random split vs client-grouped split

In Week 5 I used a client holdout split. Here I show the before/after by comparing a random split (optimistic) against the grouped split (honest).

In [3]:
# Load data
df = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        impressions_90d,
        avg_position_90d,
        query_char_count,
        query_token_count,
        content_visible_query_count,
        rare_query_count,
        rare_impressions_share,
        anonymized_impressions_share,
        impressions_last30,
        clicks_last30,
        clicks_prev30,
        (clicks_last30 > clicks_prev30) as is_rising
    FROM read_parquet('{rel}/fact_content_query_90d.parquet')
    WHERE (impressions_last30 >= 10) IS TRUE
""").df()

safe_features = [
    'impressions_90d', 'avg_position_90d', 'query_char_count',
    'query_token_count', 'content_visible_query_count',
    'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share'
]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# RANDOM SPLIT
from sklearn.model_selection import train_test_split
X = df[safe_features].fillna(0)
y = df['is_rising'].astype(int)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.2, random_state=42)

rf_r = RandomForestClassifier(n_estimators=100, random_state=42)
rf_r.fit(X_train_r, y_train_r)
random_p20 = precision_at_k(rf_r.predict_proba(X_test_r)[:,1], y_test_r, 20)

# GROUPED SPLIT (client holdout)
clients = df['client_hash_id'].unique()
np.random.shuffle(clients)
train_clients = clients[:int(len(clients)*0.8)]
test_clients = clients[int(len(clients)*0.8):]

train_df = df[df['client_hash_id'].isin(train_clients)]
test_df = df[df['client_hash_id'].isin(test_clients)]

X_train_g = train_df[safe_features].fillna(0)
y_train_g = train_df['is_rising'].astype(int)
X_test_g = test_df[safe_features].fillna(0)
y_test_g = test_df['is_rising'].astype(int)

rf_g = RandomForestClassifier(n_estimators=100, random_state=42)
rf_g.fit(X_train_g, y_train_g)
grouped_p20 = precision_at_k(rf_g.predict_proba(X_test_g)[:,1], y_test_g, 20)

print("Split comparison        Precision@20")
print("Base rate              ", round(y.mean(), 3))
print("Random split           ", round(random_p20, 3))
print("Grouped (client) split ", round(grouped_p20, 3))
print("Gap                    ", round(random_p20 - grouped_p20, 3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Split comparison        Precision@20
Base rate               0.066
Random split            0.85
Grouped (client) split  0.2
Gap                     0.65


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Leakage audit findings:

The gap between random split (0.85) and grouped split (0.20) is 0.65 — this is a major signal that the model was memorizing client-level patterns in the random split. The grouped split is the honest number.

Feature audit — all safe features confirmed:

impressions_90d — total impressions, no click data, safe
avg_position_90d — search position, no click data, safe
query_char_count / query_token_count — query shape, safe
content_visible_query_count — page-level query count, safe
rare_query_count / rare_impressions_share — query mix, safe
anonymized_impressions_share — query mix, safe

No product flags used. No future window columns used. Leakage audit: PASSED.

In [8]:
# Add leaky column and re-split
df['leaky'] = (df['clicks_last30'] - df['clicks_prev30'])

train_df2 = df[df['client_hash_id'].isin(train_clients)]
test_df2 = df[df['client_hash_id'].isin(test_clients)]

leaky_features = safe_features + ['leaky']
X_train_l = train_df2[leaky_features].fillna(0)
X_test_l = test_df2[leaky_features].fillna(0)

rf_l = RandomForestClassifier(n_estimators=100, random_state=42)
rf_l.fit(X_train_l, train_df2['is_rising'].astype(int))
leaky_p20 = precision_at_k(rf_l.predict_proba(X_test_l)[:,1], y_test_g, 20)

print("Leakage test:")
print("Honest features Precision@20:", round(grouped_p20, 3))
print("With leaky feature Precision@20:", round(leaky_p20, 3))
print("Score jumped — leaky feature confirmed, now removed")s

Leakage test:
Honest features Precision@20: 0.2
With leaky feature Precision@20: 1.0
Score jumped — leaky feature confirmed, now removed


**4. Claim rewrite**

Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.




Claim rewrite — from bold to honest:

Original bold claim (unsafe):
"Random Forest predicts rising query-page pairs with Precision@20 = 0.85, beating the baseline."

Rewritten in safe language:
"Under a random split, the Random Forest achieved Precision@20 = 0.85 — however, this figure reflects client-level memorization, not generalizable skill. Under a grouped client holdout (the honest split), Precision@20 drops to 0.20, below the impressions baseline of 0.30. This suggests that safe query-level features alone are insufficient to predict click momentum across unseen clients. The baseline remains the stronger decision-support tool at this stage."

What this means in practice:
A content team using this model on new clients would observe performance closer to 0.20 than 0.85. The honest recommendation is to use the impressions-based baseline queue until stronger time-aware features can be engineered safely.

In [9]:
print("Claim rewrite: DONE")
print("Random split P@20: 0.85 — was memorizing client patterns")
print("Grouped split P@20: 0.20 — honest generalization estimate")
print("Leaky feature P@20: 1.0 — confirmed leakage, removed")
print("Safe claim: directional, decision-support, observed only")

Claim rewrite: DONE
Random split P@20: 0.85 — was memorizing client patterns
Grouped split P@20: 0.20 — honest generalization estimate
Leaky feature P@20: 1.0 — confirmed leakage, removed
Safe claim: directional, decision-support, observed only


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.